# KNN imputer (k=5)
        **unsupervised › imputation** · notebook `knn5`

        **Kapan dipakai:** Baris mirip punya nilai mirip; menjaga struktur lokal. Data sudah dalam skala sebanding.

        **Jebakan umum:** Wajib skala seragam; lambat untuk data besar.

**Data demo:** Wine dengan 20% nilai dihapus acak (MCAR) secara buatan. Karena nilai asli diketahui, akurasi imputasi bisa diukur langsung.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
PATH = ROOT / "data_latihan/unsupervised/wine.csv"; LABEL_COL = "cultivar"; FRAC = 0.2
MODEL_KEY = "knn5"
OUT = out_dir("unsupervised", "imputation", MODEL_KEY)
SEED = 42; set_seed(SEED)

## 2. Buat data bolong (MCAR) lalu pisahkan train/test baris
Imputer dilatih HANYA pada baris train; baris test diisi memakai statistik train.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from superdb.unsupervised import mask_mcar, impute_nrmse
df = pd.read_csv(PATH); y = df[LABEL_COL].to_numpy(); X = df.drop(columns=[LABEL_COL]).select_dtypes("number").to_numpy(float)
idx_tr, idx_te = train_test_split(np.arange(len(X)), test_size=0.3, random_state=SEED, stratify=y)
sc = StandardScaler().fit(X[idx_tr]); Xs = sc.transform(X)                 # skala sebanding (penting untuk KNN)
Xtr_true, Xte_true = Xs[idx_tr], Xs[idx_te]
Xtr_m, mtr = mask_mcar(Xtr_true, FRAC, SEED); Xte_m, mte = mask_mcar(Xte_true, FRAC, SEED + 1)
print(f"{mtr.mean():.0%} sel dihapus | baris train tanpa NaN sama sekali: {int((~np.isnan(Xtr_m).any(1)).sum())} dari {len(Xtr_m)}")

## 3. Imputer

In [ ]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=5)

## 4. Akurasi imputasi pada sel yang dihapus (data test)
**nRMSE**: 0 = sempurna; ≈1 = sama buruk dengan menebak rata-rata.

In [ ]:
import time; t0 = time.time(); imputer.fit(Xtr_m); sec = time.time() - t0
Xte_i = imputer.transform(Xte_m); Xtr_i = imputer.transform(Xtr_m)
k_orig = Xte_true.shape[1]; Xte_i_o, Xtr_i_o = Xte_i[:, :k_orig], Xtr_i[:, :k_orig]     # buang kolom penanda bila ada
nr = impute_nrmse(Xte_true, Xte_i_o, mte); print(f"nRMSE = {nr:.3f}  (waktu fit {sec:.1f} dtk)")

## 5. Dampak pada model akhir
Akhirnya yang penting: apakah klasifikasi membaik? Dibandingkan dengan (a) tanpa missing (batas atas) dan (b) membuang baris bolong.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
fit_eval = lambda A, B: f1_score(y[idx_te], LogisticRegression(max_iter=3000).fit(A, y[idx_tr]).predict(B), average="macro")
f1_imp = fit_eval(Xtr_i, Xte_i); f1_oracle = fit_eval(Xtr_true, Xte_true)
keep = ~np.isnan(Xtr_m).any(1)
f1_drop = f1_score(y[idx_te], LogisticRegression(max_iter=3000).fit(Xtr_m[keep], y[idx_tr][keep]).predict(np.nan_to_num(Xte_m)), average="macro") if keep.sum() > 20 else float("nan")
display(pd.DataFrame({"F1 macro": {"data lengkap (batas atas)": f1_oracle, f"imputasi {MODEL_KEY}": f1_imp, "buang baris bolong": f1_drop}}).round(4))
print(f"baris yang tersisa bila membuang baris bolong: {int(keep.sum())} dari {len(keep)}")
save_json({"nrmse": nr, "f1_imputed": f1_imp, "f1_complete": f1_oracle, "f1_drop_rows": f1_drop, "fit_seconds": sec, "rows_kept_if_dropped": int(keep.sum())}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan imputer lain di folder ini lewat nRMSE dan F1 akhir.
- Missing yang tidak acak (MNAR) di data nyata bisa informatif: tambahkan kolom penanda (`mean_indicator`).
- Di Pipeline lomba: imputer ada di dalam Pipeline agar hanya belajar dari data training.